# mimic-video Network Architecture Diagram (Clean)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ─── Color palette ────────────────────────────────────────────────────────────
# Stage 1: blues
S1_BG    = '#EBF5FB'   # section bg
S1_BOX   = '#AED6F1'   # main boxes
S1_DARK  = '#1A5276'   # borders / arrows
S1_MID   = '#2980B9'   # secondary
# Stage 2: reds
S2_BG    = '#FDEDEC'
S2_BOX   = '#F1948A'
S2_DARK  = '#7B241C'
S2_MID   = '#C0392B'
# Neutral
IO_BOX   = '#F2F3F4'
IO_DARK  = '#2C3E50'
OUT_BOX  = '#D5F5E3'
OUT_DARK = '#1E8449'
BG       = '#FDFEFE'

# ─── Helpers ──────────────────────────────────────────────────────────────────
def box(ax, x, y, w, h, title, sub='', fc=S1_BOX, ec=S1_DARK,
        fs=8.5, bold=False, lw=1.5, alpha=1.0):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.008',
                        facecolor=fc, edgecolor=ec, linewidth=lw,
                        zorder=3, alpha=alpha)
    ax.add_patch(r)
    fw = 'bold' if bold else 'normal'
    dy = 0.013 if sub else 0
    ax.text(x+w/2, y+h/2+dy, title, ha='center', va='center',
            fontsize=fs, fontweight=fw, color=ec, zorder=4, linespacing=1.3)
    if sub:
        ax.text(x+w/2, y+h/2-0.018, sub, ha='center', va='center',
                fontsize=fs-1.5, color='#555', zorder=4, style='italic')

def section(ax, x, y, w, h, label, fc, ec, fs=8):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.01',
                        facecolor=fc, edgecolor=ec, linewidth=1.2,
                        linestyle='--', zorder=1, alpha=0.3)
    ax.add_patch(r)
    ax.text(x+0.012, y+h-0.012, label, ha='left', va='top',
            fontsize=fs, color=ec, style='italic', zorder=2, fontweight='bold')

def arrow(ax, x0, y0, x1, y1, ec=IO_DARK, label='', lbl_x=None, lbl_y=None,
          lw=1.6, style='->', ls='-'):
    ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle=style, color=ec, lw=lw,
                                mutation_scale=11,
                                connectionstyle='arc3,rad=0'),
                zorder=5)
    if label:
        lx = lbl_x if lbl_x is not None else (x0+x1)/2+0.01
        ly = lbl_y if lbl_y is not None else (y0+y1)/2
        ax.text(lx, ly, label, fontsize=7, color=ec, ha='left', va='center',
                bbox=dict(fc='white', ec='none', alpha=0.85, pad=1), zorder=6)

# ─── Canvas ───────────────────────────────────────────────────────────────────
W, H = 18, 10
fig, ax = plt.subplots(figsize=(W, H))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# ═══════════════════════════════════════════════════════════════════════════════
#  LAYOUT  (left → right, each column x-center)
#  Col A (inputs):  x ∈ [0.02, 0.13]
#  Col B (Stage1):  x ∈ [0.17, 0.50]
#  Col C (bridge):  x ∈ [0.52, 0.60]   ← h_video
#  Col D (Stage2):  x ∈ [0.62, 0.87]
#  Col E (output):  x ∈ [0.89, 0.98]
# ═══════════════════════════════════════════════════════════════════════════════

# ─── Section backgrounds ──────────────────────────────────────────────────────
# Stage 1
section(ax, 0.14, 0.10, 0.40, 0.86, 'Stage 1  —  CosmosVideoBackbone',
        S1_BG, S1_DARK, fs=9)
# Stage 2
section(ax, 0.56, 0.10, 0.33, 0.86, 'Stage 2  —  ActionDecoderDiT',
        S2_BG, S2_DARK, fs=9)

# ─── Column A: Inputs ─────────────────────────────────────────────────────────
# video frames
box(ax, 0.02, 0.790, 0.115, 0.07,
    'Video Frames', '3 cams × 17 frames\n[B, 3, 17, 480, 640]',
    IO_BOX, IO_DARK, fs=7.5, bold=True)
# text prompt
box(ax, 0.02, 0.695, 0.115, 0.07,
    'Text Prompt', '"Organize table…"',
    IO_BOX, IO_DARK, fs=7.5, bold=True)
# noisy action
box(ax, 0.02, 0.490, 0.115, 0.07,
    'Noisy Actions  ε', '[B, 16, 16]',
    IO_BOX, IO_DARK, fs=7.5, bold=True)
# proprio
box(ax, 0.02, 0.395, 0.115, 0.07,
    'Proprioception', '[B, 16]',
    IO_BOX, IO_DARK, fs=7.5, bold=True)
# timesteps
box(ax, 0.02, 0.300, 0.115, 0.07,
    'Timesteps', 'τ_v,  τ_a  ∈ [0,1]',
    IO_BOX, IO_DARK, fs=7.5, bold=True)

# ─── Column B: Stage 1 ────────────────────────────────────────────────────────
# VAE Encoder
box(ax, 0.165, 0.790, 0.20, 0.07,
    'Cosmos VAE Encoder  [frozen]', 'z  =  [B, 16, 5, 30, 40]',
    S1_BOX, S1_DARK, fs=8, bold=True)

# T5 Encoder
box(ax, 0.165, 0.695, 0.20, 0.07,
    'T5 Text Encoder  [frozen]', 'emb  =  [B, 512, 4096]',
    S1_BOX, S1_DARK, fs=8, bold=True)

# Divider
ax.plot([0.14, 0.54], [0.660, 0.660], color=S1_DARK, lw=0.6, ls=':', zorder=2, alpha=0.5)

# Cosmos DiT body — big main box
box(ax, 0.165, 0.340, 0.385, 0.295,
    '', '',
    S1_BOX, S1_DARK, fs=9, lw=2.0, alpha=0.45)
ax.text(0.358, 0.624, 'Cosmos 2B Video DiT', ha='center', va='top',
        fontsize=10, fontweight='bold', color=S1_DARK, zorder=5)
ax.text(0.358, 0.608, '28 Transformer Blocks  ·  T5 Cross-Attention  ·  LoRA (rank=16)',
        ha='center', va='top', fontsize=7.5, color=S1_MID, zorder=5)

# Sub-boxes inside DiT
for i, (lbl, sub) in enumerate([
    ('Patchify\n+ PosEmbed',   ''),
    ('Blocks 0-18\n(AdaNorm + Attn)', '19 blocks'),
    ('Block 19\n(Hook  →  h_video)',  '[B,T·H·W,2048]'),
    ('Blocks 20-27', '8 blocks'),
]):
    bx = 0.175 + i * 0.093
    box(ax, bx, 0.360, 0.082, 0.215,
        lbl, sub, S1_BOX, S1_DARK, fs=7.5, lw=0.9)

# LoRA badge
box(ax, 0.175, 0.270, 0.18, 0.055,
    'LoRA  (rank / α = 16 / 16)', 'attn1/2.to_q/k/v/out;  ff.net',
    '#D6EAF8', S1_DARK, fs=7.5, lw=1)

# Spatial pooling
box(ax, 0.38, 0.270, 0.165, 0.055,
    'Spatial Pool', 'mean  →  [B, 5, 2048]\nnone   →  [B, 1500, 2048]',
    '#D6EAF8', S1_DARK, fs=7.5, lw=1)

# Arrow: hook output → pool
arrow(ax, 0.347, 0.360, 0.347, 0.325, S1_DARK, label='h_video raw',
      lbl_x=0.352, lbl_y=0.340)

# Arrow: video → VAE
arrow(ax, 0.135, 0.825, 0.165, 0.825, S1_MID)
# Arrow: text → T5
arrow(ax, 0.135, 0.730, 0.165, 0.730, S1_MID)
# VAE → DiT
arrow(ax, 0.265, 0.790, 0.265, 0.635, S1_MID, label='z',
      lbl_x=0.270, lbl_y=0.712)
# T5 → DiT (cross-attn)
arrow(ax, 0.310, 0.695, 0.310, 0.635, S1_MID, label='encoder_hidden_states',
      lbl_x=0.315, lbl_y=0.662)

# ─── Bridge: h_video → Stage 2  ───────────────────────────────────────────────
# Arrow: pool → right edge → Stage 2 cross-attn
arrow(ax, 0.545, 0.298, 0.620, 0.298, OUT_DARK,
      label='h_video  [B, T, 2048]', lbl_x=0.549, lbl_y=0.308)

# arrow from pool out (right)
arrow(ax, 0.478-0.04+0.04, 0.297, 0.545, 0.297, S1_MID)

# ─── Column D: Stage 2 ────────────────────────────────────────────────────────
# Input projections
box(ax, 0.620, 0.790, 0.185, 0.055,
    'Action Input MLP', '[B,16,16]  →  [B,16,512]',
    S2_BOX, S2_DARK, fs=7.5)
box(ax, 0.620, 0.720, 0.185, 0.055,
    'Proprio Input MLP', '[B,16]  →  [B,1,512]',
    S2_BOX, S2_DARK, fs=7.5)
box(ax, 0.620, 0.640, 0.185, 0.055,
    'Concat  +  Pos Embed', '[B, 17, 512]',
    S2_BOX, S2_DARK, fs=7.5)

# Timestep embed
box(ax, 0.620, 0.555, 0.185, 0.065,
    'Bilinear Timestep Embed', 'SinEmbed(τ_v) ⊙ SinEmbed(τ_a)\n→  cond  [B, 512]',
    '#FDEBD0', S2_DARK, fs=7.5)

# 8 Decoder blocks
box(ax, 0.620, 0.200, 0.26, 0.335,
    '', '', S2_BOX, S2_DARK, lw=2.0, alpha=0.4)
ax.text(0.75, 0.525, '× 8  ActionDecoderBlocks', ha='center', va='top',
        fontsize=10, fontweight='bold', color=S2_DARK, zorder=5)
# three sub-boxes inside blocks
for i, (lbl, sub) in enumerate([
    ('Cross-Attention', 'Q=actions\nKV=h_video'),
    ('Self-Attention', 'actions × actions'),
    ('AdaLN-Zero\n+ MLP (GELU)', 'cond modulated'),
]):
    bx = 0.630 + i*0.086
    box(ax, bx, 0.215, 0.076, 0.275,
        lbl, sub, S2_BOX, S2_DARK, fs=7.5, lw=0.9)

# Output head
box(ax, 0.620, 0.130, 0.185, 0.055,
    'LayerNorm  +  Linear  (zero-init)', 'velocity  [B, 16, 16]',
    OUT_BOX, OUT_DARK, fs=7.5)

# Arrows inside Stage 2
# Action / Proprio → input MLPs
arrow(ax, 0.135, 0.525, 0.620, 0.818, S2_MID)
arrow(ax, 0.135, 0.430, 0.620, 0.748, S2_MID)
# τ → timestep embed
arrow(ax, 0.135, 0.335, 0.620, 0.587, S2_MID,
      label='τ_v, τ_a', lbl_x=0.360, lbl_y=0.430)
# Action MLP → concat
arrow(ax, 0.713, 0.790, 0.713, 0.695, S2_MID)
# Proprio MLP → concat
arrow(ax, 0.713, 0.720, 0.713, 0.695, S2_MID)
# concat → blocks
arrow(ax, 0.713, 0.640, 0.713, 0.535, S2_MID)
# timestep → blocks (into cond)
arrow(ax, 0.713, 0.555, 0.713, 0.535, S2_MID)
# combined → blocks
arrow(ax, 0.713, 0.535, 0.713, 0.490, S2_MID)
# h_video → cross-attn in blocks
arrow(ax, 0.620, 0.298, 0.630+0.038, 0.490, OUT_DARK,
      label='', lbl_x=0.610, lbl_y=0.390)
# blocks → output head
arrow(ax, 0.713, 0.200, 0.713, 0.185, S2_MID)

# ─── Column E: Output ─────────────────────────────────────────────────────────
box(ax, 0.895, 0.430, 0.095, 0.095,
    'Action Chunk', '16 steps\n× 16 DOF',
    OUT_BOX, OUT_DARK, fs=8, bold=True)
arrow(ax, 0.805, 0.157, 0.895, 0.478, OUT_DARK,
      label='Euler ODE\n(10 steps, τ:1→0)', lbl_x=0.813, lbl_y=0.280)

# ─── Title ────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975, 'mimic-video  ·  Full Pipeline Architecture',
        ha='center', va='top', fontsize=15, fontweight='bold',
        color='#1A252F', transform=ax.transAxes)
ax.text(0.50, 0.952,
        'Stage 1 (blue): Cosmos 2B DiT + LoRA, extracts h_video at layer 19    ·    '
        'Stage 2 (red): ActionDecoderDiT trained with flow matching',
        ha='center', va='top', fontsize=8.5, color='#555', transform=ax.transAxes)

# ─── Legend ───────────────────────────────────────────────────────────────────
legend_items = [
    (IO_BOX,   IO_DARK,  'Inputs'),
    (S1_BOX,   S1_DARK,  'Stage 1 (Video Backbone)'),
    (S2_BOX,   S2_DARK,  'Stage 2 (Action Decoder)'),
    (OUT_BOX,  OUT_DARK, 'Output'),
    ('#FDEBD0', S2_DARK, 'Flow Matching'),
]
base_x = 0.15
for i, (fc, ec, lbl) in enumerate(legend_items):
    px = base_x + i * 0.165
    r = FancyBboxPatch((px, 0.01), 0.018, 0.030,
                        boxstyle='round,pad=0.003', facecolor=fc, edgecolor=ec,
                        linewidth=1.2, transform=ax.transAxes, zorder=6)
    ax.add_patch(r)
    ax.text(px+0.023, 0.025, lbl, va='center', fontsize=8,
            color='#333', transform=ax.transAxes)

plt.tight_layout(rect=[0, 0.05, 1, 0.945])
plt.savefig('architecture.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → architecture.png')

In [3]:
import wandb
import pandas as pd

# 配置项目信息
ENTITY = "qiang"
PROJECT = "dit4dit-stage1-object"
#
api = wandb.Api()

# 1. (可选) 列出项目下所有的 Runs，方便查找 Run ID
print(f"--- 项目 {ENTITY}/{PROJECT} 下的运行列表 ---")
runs = api.runs(f"{ENTITY}/{PROJECT}")
for r in runs:
    print(f"Run Name: {r.name} | Run ID: {r.id} | State: {r.state}")

# 2. 填写你想提取的具体的 Run ID (从上面列表或网页 URL 中获取)
# 例如: run_id = "abcd1234"
run_id = input("\n请输入你要提取的 Run ID: ")

try:
    target_run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    print(f"\n正在提取 {target_run.name} ({run_id}) 的数据...")
    
    # 3. 提取 train/loss 和 step
    # samples 参数设大一点（例如 100000）以确保获取全部历史数据
    history = target_run.history(keys=["train/loss", "train/step"], pandas=True)
    
    if history.empty:
        print("未找到数据，请确认 Key 是否正确 (默认应为 'train/loss')")
    else:
        # 4. 排序并保存
        history = history.sort_values("train/step")
        filename = f"stage2_loss_{run_id}.csv"
        history.to_csv(filename, index=False)
        print(f"提取完成！数据已保存至 {filename}")
        print(history.head()) # 显示前几行预览

except Exception as e:
    print(f"发生错误: {e}")


--- 项目 qiang/dit4dit-stage1-object 下的运行列表 ---
Run Name: stage1-libero_object | Run ID: wyyakg3u | State: crashed

正在提取 stage1-libero_object (wyyakg3u) 的数据...
提取完成！数据已保存至 stage2_loss_wyyakg3u.csv
   _step  train/loss  train/step
0     10    0.240481          10
1     20    0.251641          20
2     30    0.247644          30
3     40    0.245643          40
4     50    0.251581          50
